In [1]:
SYMBOL = "BTCUSDT"
TARGET_HORIZON = 5
MODEL_TYPE = "rf"

In [2]:
# Parameters
SYMBOL = "ADAUSDT"
TARGET_HORIZON = 5
MODEL_TYPE = "xgb"


In [3]:
import os
import time
import json
import joblib
import pandas as pd
import numpy as np
import optuna
from functools import partial
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from features import add_features
from constants import DATA_DIR, MODEL_DIR
from utils import time_split, information_coefficient, rank_information_coefficient
from models import OBJECTIVES, MODEL_REGISTRY

/home/rachmiel/quant/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
MODEL_DIR = os.path.join(MODEL_DIR, MODEL_TYPE)
PARQUET_PATH = f"{DATA_DIR}/{SYMBOL}_1m.parquet"

os.makedirs(MODEL_DIR, exist_ok=True)

In [5]:
model_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_model.joblib")
features_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_cols.json")
meta_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_meta.json")
fi_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_importance.csv")
pred_path = os.path.join(MODEL_DIR, f"{SYMBOL}__{TARGET_HORIZON}_predictions.csv")

In [6]:
df = pd.read_parquet(PARQUET_PATH)
print(f"[info] raw rows: {len(df):,}")

# add features + target
df, feature_cols = add_features(df, TARGET_HORIZON)

[info] raw rows: 284,679


In [7]:
df.head()

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,num_trades,taker_buy_base_asset_volume,...,dow_cos,dom_sin,dom_cos,month_sin,month_cos,macd,macd_signal,macd_hist,atr_14,atr_norm
0,2025-09-01 00:00:00+00:00,0.8112,0.8112,0.8102,0.8112,149838.6,2025-09-01 00:00:59.999999+00:00,121464.19529,305,51474.3,...,1.0,0.201299,0.97953,-1.0,-1.836970e-16,0.000000,0.000000,0.000000,NaN,NaN
1,2025-09-01 00:01:00+00:00,0.8112,0.8119,0.8110,0.8118,97007.1,2025-09-01 00:01:59.999999+00:00,78722.09451,184,66919.4,...,1.0,0.201299,0.97953,-1.0,-1.836970e-16,0.000013,0.000007,0.000006,NaN,NaN
2,2025-09-01 00:02:00+00:00,0.8118,0.8119,0.8106,0.8111,56191.5,2025-09-01 00:02:59.999999+00:00,45580.17305,187,13938.4,...,1.0,0.201299,0.97953,-1.0,-1.836970e-16,-0.000005,0.000003,-0.000007,NaN,NaN
3,2025-09-01 00:03:00+00:00,0.8112,0.8116,0.8108,0.8108,56303.8,2025-09-01 00:03:59.999999+00:00,45665.54211,160,16397.3,...,1.0,0.201299,0.97953,-1.0,-1.836970e-16,-0.000023,-0.000006,-0.000017,NaN,NaN
4,2025-09-01 00:04:00+00:00,0.8107,0.8107,0.8075,0.8077,375975.0,2025-09-01 00:04:59.999999+00:00,304105.33884,977,110194.0,...,1.0,0.201299,0.97953,-1.0,-1.836970e-16,-0.000157,-0.000051,-0.000106,NaN,NaN


In [8]:
target_col = f"target_ret_fwd_{TARGET_HORIZON}"

model_df = df[["open_time"] + feature_cols + [target_col]].copy()

# Remove:
# early rows where rolling features don’t exist yet
# rows where z-scores / ratios blew up
# rows where target is NaN (due to future shift)
model_df = model_df.replace([np.inf, -np.inf], np.nan)
model_df = model_df.dropna(subset=feature_cols + [target_col])

print(f"[info] usable rows after features: {len(model_df):,}")

train_df, test_df = time_split(model_df, train_frac=0.8)

# Further split the training set into train/valid for Optuna
optuna_train_df, valid_df = time_split(train_df, train_frac=0.8)

X_train = optuna_train_df[feature_cols]
y_train = optuna_train_df[target_col]

X_valid = valid_df[feature_cols]
y_valid = valid_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

train_start_time = pd.to_datetime(train_df["open_time"].iloc[0], utc=True)
train_end_time = pd.to_datetime(train_df["open_time"].iloc[-1], utc=True)

val_start_time = pd.to_datetime(valid_df["open_time"].iloc[0], utc=True)
val_end_time = pd.to_datetime(valid_df["open_time"].iloc[-1], utc=True)

test_start_time = pd.to_datetime(test_df["open_time"].iloc[0], utc=True)
test_end_time = pd.to_datetime(test_df["open_time"].iloc[-1], utc=True)

print(f"[info] optuna train rows: {len(optuna_train_df):,}")
print(f"[info] valid rows:        {len(valid_df):,}")
print(f"[info] test rows:         {len(test_df):,}")

[info] usable rows after features: 284,331
[info] optuna train rows: 181,971
[info] valid rows:        45,493
[info] test rows:         56,867


In [9]:
study = optuna.create_study(direction="maximize")
objective_fn = partial(
    OBJECTIVES[MODEL_TYPE],
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
)

study.optimize(objective_fn, n_trials=50, show_progress_bar=True)

print("\n[optuna] best trial")
print(f"value: {study.best_value:.6f}")
print("params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

[I 2026-03-19 22:13:40,557] A new study created in memory with name: no-name-bcc37389-b06b-4428-884f-ea4fac7af261


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:03<?, ?it/s]

Best trial: 0. Best value: 0.0153051:   0%|          | 0/50 [00:03<?, ?it/s]

Best trial: 0. Best value: 0.0153051:   2%|▏         | 1/50 [00:03<02:39,  3.25s/it]

...

Best trial: 40. Best value: 0.0327682:  98%|█████████▊| 49/50 [06:41<00:07,  7.84s/it]

Best trial: 40. Best value: 0.0327682:  98%|█████████▊| 49/50 [06:41<00:07,  7.84s/it]

Best trial: 40. Best value: 0.0327682: 100%|██████████| 50/50 [06:41<00:00,  6.95s/it]

Best trial: 40. Best value: 0.0327682: 100%|██████████| 50/50 [06:41<00:00,  8.03s/it]

[I 2026-03-19 22:20:21,883] Trial 49 finished with value: 0.0278589884094813 and parameters: {'n_estimators': 400, 'max_depth': 12, 'learning_rate': 0.03859303064843673, 'subsample': 0.9535811469918248, 'colsample_bytree': 0.8100421511442708, 'min_child_weight': 8, 'reg_alpha': 8.749611491280137e-07, 'reg_lambda': 3.307141250014575e-07}. Best is trial 40 with value: 0.03276816604557459.

[optuna] best trial
value: 0.032768
params:
  n_estimators: 600
  max_depth: 12
  learning_rate: 0.03470825372126559
  subsample: 0.9988998989844804
  colsample_bytree: 0.980730981599268
  min_child_weight: 2
  reg_alpha: 2.9200262734769805e-08
  reg_lambda: 3.9458246593867246e-07


In [10]:
best_params = study.best_params.copy()
best_params["random_state"] = 42
best_params["n_jobs"] = -1

X_train_full = train_df[feature_cols]
y_train_full = train_df[target_col]

final_model = MODEL_REGISTRY[MODEL_TYPE](**best_params)

start = time.time()
print(f"[training] fitting final {MODEL_TYPE}...")
final_model.fit(X_train_full, y_train_full)
print(f"[training] done in {time.time() - start:.2f}s")

[training] fitting final xgb...


[training] done in 286.36s


In [11]:
train_pred = final_model.predict(X_train_full)
test_pred = final_model.predict(X_test)

In [12]:
# evaluate
print("[eval] computing metrics...")
train_ic = information_coefficient(y_train_full.values, train_pred)
test_ic = information_coefficient(y_test.values, test_pred)

train_rank_ic = rank_information_coefficient(y_train_full.values, train_pred)
test_rank_ic = rank_information_coefficient(y_test.values, test_pred)

train_rmse = root_mean_squared_error(y_train_full, train_pred)
test_rmse = root_mean_squared_error(y_test, test_pred)

print("\n===== RESULTS =====")
print(f"Train IC:      {train_ic:.6f}")
print(f"Test IC:       {test_ic:.6f}")
print(f"Train Rank IC: {train_rank_ic:.6f}")
print(f"Test Rank IC:  {test_rank_ic:.6f}")
print(f"Train RMSE:    {train_rmse:.6f}")
print(f"Test RMSE:     {test_rmse:.6f}")

[eval] computing metrics...

===== RESULTS =====
Train IC:      0.960759
Test IC:       0.020662
Train Rank IC: 0.934148
Test Rank IC:  0.029694
Train RMSE:    0.001104
Test RMSE:     0.002901


In [13]:
# feature importance
importances = pd.Series(
    final_model.feature_importances_,
    index=feature_cols
).sort_values(ascending=False)

print("\n===== FEATURE IMPORTANCE =====")
print(importances)


===== FEATURE IMPORTANCE =====
vol_5               0.141522
trend_x_imb         0.080612
trend_strength      0.078475
hour_sin            0.044289
volume_mom_5        0.033527
imbalance_15        0.032791
vol_regime_ratio    0.030972
hour_cos            0.030865
mom_3               0.028513
vol_15              0.024823
dom_sin             0.023412
range_ratio         0.022876
mom_30              0.022813
vol_ratio_5_30      0.022641
trades_z            0.021100
vol_30              0.020773
imbalance_5         0.019084
num_trades_mom_5    0.019050
mr_x_vol            0.017333
dist_ma_5           0.016360
imbalance           0.016264
dist_ma_15_z        0.015116
is_trending         0.014467
month_sin           0.014275
mom_x_imb           0.013393
mom_15              0.013381
dow_sin             0.013105
range_15            0.013063
month_cos           0.013008
dow_cos             0.012778
atr_norm            0.012731
is_high_vol         0.012578
macd_hist           0.012551
mom_60     

In [14]:
# save predictions
out = test_df[["open_time", target_col]].copy()
out["prediction"] = test_pred
out.to_csv(pred_path, index=False)
print(f"\n[saved] predictions -> {pred_path}")


[saved] predictions -> models/xgb/ADAUSDT__5_predictions.csv


In [15]:
# save model
joblib.dump(final_model, model_path)

# save feature columns
with open(features_path, "w") as f:
    json.dump(feature_cols, f, indent=2)

# save feature importance
importances.to_csv(fi_path, header=["importance"])

# save metadata
meta = {
    "symbol": SYMBOL,
    "target_horizon": int(TARGET_HORIZON),
    "target_col": target_col,
    "model_type": MODEL_TYPE,
    "study_best_value": float(study.best_value),
    "model_params": best_params,
    "n_features": int(len(feature_cols)),
    "feature_cols_path": str(features_path),
    "model_path": str(model_path),
    "feature_importance_path": str(fi_path) if fi_path is not None else None,
    "train_ic": train_ic,
    "test_ic": test_ic,
    "train_rank_ic": train_rank_ic,
    "test_rank_ic": test_rank_ic,
    "train_rmse": train_rmse,
    "test_rmse": test_rmse,
    "train_start_time": pd.Timestamp(train_start_time).isoformat(),
    "train_end_time": pd.Timestamp(train_end_time).isoformat(),
    "val_start_time": pd.Timestamp(val_start_time).isoformat(),
    "val_end_time": pd.Timestamp(val_end_time).isoformat(),
    "test_start_time": pd.Timestamp(test_start_time).isoformat(),
    "test_end_time": pd.Timestamp(test_end_time).isoformat()
}

with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print(f"[saved] model -> {model_path}")
print(f"[saved] features -> {features_path}")
print(f"[saved] feature importance -> {fi_path}")
print(f"[saved] metadata -> {meta_path}")

[saved] model -> models/xgb/ADAUSDT__h5_model.joblib
[saved] features -> models/xgb/ADAUSDT__h5_feature_cols.json
[saved] feature importance -> models/xgb/ADAUSDT__h5_feature_importance.csv
[saved] metadata -> models/xgb/ADAUSDT__h5_meta.json
